<a href="https://colab.research.google.com/github/koushalkarthik15/mlflyrankkarthik/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/koushalkarthik15/mlflyrankkarthik/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

For Lane 2 (Opportunity Scoring), the unit of analysis is **one content item (page) per day**, aggregated up to the item level. I am using the `fact_content_daily_performance` table joined with `dim_content`. My time window is a single mid-panel month: **March 2026 (`month=2026-03`)**.






In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

*   **Features:** `imp_first_half`, `clicks_first_half`, `pos_first_half`, `word_count`, `content_type`. These are knowable at the decision moment because they only use strictly historical data (the first 15 days of the month).
*   **Label / Proxy:** `imp_second_half` (impressions in the second half of the month). This is the future outcome we are trying to predict.
*   **Context:** `content_hash_id` (used only to group and join the data, never given to the model to learn from).
*   **Excluded:** Any derived column that looks at the future (like our `TRAP_leaky_trend_pct`). Including this would cause massive data leakage because the model would be cheating by looking at the second half of the month!


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

!pip install duckdb huggingface_hub

import duckdb
import pandas as pd
from google.colab import userdata

# 1. Setup Hugging Face access natively for DuckDB
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

print("--- Query 1: Row count, Date span, and Availability (IS TRUE) ---")
q1 = """
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as min_date,
    MAX(report_date) as max_date,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) as rows_with_ga4
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
display(con.execute(q1).df())

print("\n--- Query 2: Checking the Grain (Should return 0 rows if grain is truly page-day) ---")
q2 = """
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING c > 1
LIMIT 5
"""
display(con.execute(q2).df())

print("\n--- Query 3: Feature Frame + The Deliberate Leak Trap ---")
q3 = """
WITH windowed AS (
    SELECT
        content_hash_id,
        -- HONEST FEATURES: Only using data from the first half of the month
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_first_half,
        AVG(CASE WHEN report_date <= '2026-03-15' THEN gsc_avg_position END) AS pos_first_half,

        -- THE LABEL SOURCE: What happens in the second half of the month?
        SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    GROUP BY content_hash_id
)
SELECT
    w.content_hash_id,
    w.imp_first_half,
    w.clicks_first_half,
    w.pos_first_half,
    c.word_count,
    c.content_type,

    -- THE TRAP: A derived column using data from the FUTURE (second half of the month).
    -- If we use this to predict the second half, the model will cheat and score 99%!
    (w.imp_second_half / (w.imp_first_half + 1.0)) as TRAP_leaky_trend_pct
FROM windowed w
JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
  ON w.content_hash_id = c.content_hash_id
LIMIT 5
"""
display(con.execute(q3).df())


--- Query 1: Row count, Date span, and Availability (IS TRUE) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,min_date,max_date,rows_with_ga4
0,9841378,2026-03-01,2026-03-31,413966



--- Query 2: Checking the Grain (Should return 0 rows if grain is truly page-day) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c



--- Query 3: Feature Frame + The Deliberate Leak Trap ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,imp_first_half,clicks_first_half,pos_first_half,word_count,content_type,TRAP_leaky_trend_pct
0,content_d0dff76c889de68f,111.0,0.0,5.222776,2999,keyword article,0.625000
1,content_67741cce996cfafa,38.0,1.0,4.638889,3057,keyword article,0.205128
2,content_2e6360ad20fd7107,219.0,1.0,3.737399,2855,keyword article,3.090909
3,content_ac8663da7484669a,20.0,0.0,3.597222,3281,keyword article,0.666667
4,content_65c50dfe9d87a585,1494.0,0.0,6.156643,2779,keyword article,1.079599


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Unbalanced panel history:** Because different clients started tracking their analytics at different times, we must explicitly check if `ga4_data_available IS TRUE` (Query 1 shows over 9 million rows, but only 400k have GA4 data). If we assume those nulls mean "zero engagement", we will ruin our features.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.